# Agent Design · Day 25 — **Agentic Architecture Patterns**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~2.5 hrs

Modules 1–3 gave you the parts — LangGraph, Bedrock, RAG. Module 4 is about *how an agent thinks*: the **reasoning architecture** wrapped around the LLM. Pick the wrong one and a simple lookup takes six LLM calls, or a compliance decision has no audit trail. Four patterns dominate:

- **ReAct** — interleave **Reason** and **Act** (tool calls) in a loop until done.
- **Plan-and-Solve** — plan *all* sub-steps first, then execute the plan.
- **Reflexion** — act, **self-critique** the result, retry with the lesson.
- **LATS** — tree search over ReAct trajectories (explore branches, keep the best).

Today you implement the first three, map each to a banking use case, and pick by task shape.

> Keyless. The "LLM" is a deterministic rule-based reasoner and the tools are local functions — so the **control flow** (the pattern) is what you see, not model quality. Swap in `ChatBedrock` and the loops are unchanged.

## 1. Tools + a mock reasoner

Every pattern needs tools to act on and a reasoner to decide. A tiny banking backend: look up an account's overdraft limit, and a calculator. The reasoner is deterministic (keyword→action) so runs are reproducible.

In [1]:
import re

ACCOUNTS = {
    "sarah": {"account": "AC-1001", "overdraft": 3000},
    "james": {"account": "AC-1002", "overdraft": 1500},
    "priya": {"account": "AC-1003", "overdraft": 2000},
}

def lookup_overdraft(name):
    rec = ACCOUNTS.get(name.lower().strip())
    return rec["overdraft"] if rec else None

def calculator(expr):
    if not re.fullmatch(r"[\d+\-*/ .()]+", expr): return None   # only arithmetic
    return eval(expr)                                            # safe: regex-gated

TOOLS = {"lookup_overdraft": lookup_overdraft, "calculator": calculator}
print("tools:", list(TOOLS))
print("sarah overdraft:", lookup_overdraft("sarah"))

tools: ['lookup_overdraft', 'calculator']
sarah overdraft: 3000


☝ Two tools and a keyword the reasoner can dispatch on. `calculator` is regex-gated before `eval` — never `eval` free text from a model (prompt-injection → RCE). This is the tool layer every pattern below drives; only the *reasoning loop* around it changes.

## 2. ReAct — Reason → Act → Observe, looping

**ReAct** runs a loop: the reasoner emits a **Thought** and an **Action** (tool + input); you execute it and feed the **Observation** back; repeat until it emits a final answer. The interleaving lets each step's result inform the next — the default agent loop in LangGraph and Bedrock Agents.

In [2]:
def react(question, max_steps=6, verbose=True):
    names = re.findall(r"\b(sarah|james|priya)\b", question.lower())
    trace, collected = [], []
    step = 0
    for nm in names:                       # Reason: "I need each person's overdraft"
        step += 1
        val = lookup_overdraft(nm)         # Act
        trace.append(f"Thought {step}: need {nm}'s overdraft -> Act: lookup_overdraft({nm}) -> Obs: {val}")
        collected.append(val)
    step += 1
    if "total" in question.lower() or "sum" in question.lower():
        expr = "+".join(map(str, collected))     # Reason: sum them
        total = calculator(expr)                 # Act
        trace.append(f"Thought {step}: sum exposures -> Act: calculator({expr}) -> Obs: {total}")
        answer = total
    else:
        answer = collected[0] if collected else None
    if verbose:
        for t in trace: print(t)
    return answer, trace

q = "What is the total overdraft exposure across Sarah and James accounts?"
print("Q:", q, "\n")
ans, _ = react(q)
print("\nReAct answer:", ans)

Q: What is the total overdraft exposure across Sarah and James accounts? 

Thought 1: need sarah's overdraft -> Act: lookup_overdraft(sarah) -> Obs: 3000
Thought 2: need james's overdraft -> Act: lookup_overdraft(james) -> Obs: 1500
Thought 3: sum exposures -> Act: calculator(3000+1500) -> Obs: 4500

ReAct answer: 4500


☝ ReAct discovered it needed **two lookups then a sum**, one observation feeding the next — it never planned ahead, it reasoned step-by-step. That adaptivity is ReAct's strength (handles surprises mid-task) and its weakness: on a well-understood task it spends an LLM call per step. When you *know* the steps upfront, plan instead.

## 3. Plan-and-Solve — plan all steps, then execute

**Plan-and-Solve** splits reasoning in two: first the LLM writes the **whole plan** (an ordered list of sub-tasks), then a cheap executor runs it. Fewer reasoning calls, and the plan is an **auditable artifact** before any action fires — valuable when a regulator asks *"what was the agent going to do?"*

In [3]:
def plan(question):
    names = re.findall(r"\b(sarah|james|priya)\b", question.lower())
    steps = [("lookup_overdraft", nm) for nm in names]
    if "total" in question.lower() or "sum" in question.lower():
        steps.append(("calculator", "SUM"))
    return steps

def solve(steps):
    results, trace = [], []
    for tool, arg in steps:
        if tool == "calculator" and arg == "SUM":
            expr = "+".join(map(str, results)); out = calculator(expr)
            trace.append(f"exec calculator({expr}) = {out}")
        else:
            out = TOOLS[tool](arg); results.append(out)
            trace.append(f"exec {tool}({arg}) = {out}")
    return (out, trace)

steps = plan(q)
print("PLAN (before any action):")
for i, s in enumerate(steps, 1): print(f"  {i}. {s[0]}({s[1]})")
ans, trace = solve(steps)
print("\nEXECUTE:")
for t in trace: print("  ", t)
print("\nPlan-and-Solve answer:", ans)

PLAN (before any action):
  1. lookup_overdraft(sarah)
  2. lookup_overdraft(james)
  3. calculator(SUM)

EXECUTE:
   exec lookup_overdraft(sarah) = 3000
   exec lookup_overdraft(james) = 1500
   exec calculator(3000+1500) = 4500

Plan-and-Solve answer: 4500


☝ The full plan exists *before* execution — you can log it, review it, even gate it on human approval, then run it with one reasoning call instead of one-per-step. Trade-off vs ReAct: Plan-and-Solve can't adapt if step 2's result invalidates the plan. Use it for **predictable, multi-step** tasks (reconciliations, batch checks); use ReAct when the path is unknown.

## 4. Reflexion — act, self-critique, retry

**Reflexion** adds a feedback loop: run, **critique** the output against the goal, and if it falls short, retry with the critique as extra context. It's how agents recover from their own mistakes — critical for compliance answers that must be *complete*.

In [4]:
def buggy_attempt(question):
    # simulate a first attempt that forgets one account (common agent error)
    names = re.findall(r"\b(sarah|james|priya)\b", question.lower())[:1]   # BUG: only first
    vals = [lookup_overdraft(n) for n in names]
    return sum(vals), names

def critique(question, used_names):
    wanted = set(re.findall(r"\b(sarah|james|priya)\b", question.lower()))
    missing = wanted - set(used_names)
    return missing            # empty = passed

def reflexion(question, max_tries=3):
    used = None
    for attempt in range(1, max_tries + 1):
        if used is None:
            ans, used = buggy_attempt(question)          # first (flawed) attempt
        else:
            ans, used = react(question, verbose=False)   # retry, this time complete
            used = re.findall(r"\b(sarah|james|priya)\b", question.lower())
        missing = critique(question, used)
        print(f"attempt {attempt}: answer={ans}, used={used}, "
              f"critique={'PASS' if not missing else f'missing {missing}'}")
        if not missing:
            return ans
    return ans

print("Q:", q, "\n")
final = reflexion(q)
print("\nReflexion final answer:", final)

Q: What is the total overdraft exposure across Sarah and James accounts? 

attempt 1: answer=3000, used=['sarah'], critique=missing {'james'}
attempt 2: answer=4500, used=['sarah', 'james'], critique=PASS

Reflexion final answer: 4500


☝ Attempt 1 answered **3000** — it silently dropped James. The critique caught the gap (*missing {'james'}*), and the retry produced the complete **4500**. Reflexion is the pattern for tasks where a *wrong-but-plausible* answer is dangerous: the self-check is a second line of defence the single-shot patterns lack. Cost: extra passes per task.

## 5. Choose by task shape (the ADR)

There's no best pattern — there's a fit. This is the decision table you'd put in an Architecture Decision Record.

In [5]:
DECISION = [
    ("ReAct",          "path unknown, needs tools mid-reasoning", "customer support triage, ad-hoc RAG Q&A"),
    ("Plan-and-Solve", "steps predictable, want an auditable plan","batch reconciliation, KYC checklist"),
    ("Reflexion",      "wrong answer is costly, needs completeness","compliance answers, fraud case summary"),
    ("LATS",           "high-stakes, explore alternatives, can pay","complex underwriting, multi-option planning"),
]
print(f"{'pattern':16}{'use when':44}{'banking example'}")
print("-" * 100)
for p, when, ex in DECISION:
    print(f"{p:16}{when:44}{ex}")

pattern         use when                                    banking example
----------------------------------------------------------------------------------------------------
ReAct           path unknown, needs tools mid-reasoning     customer support triage, ad-hoc RAG Q&A
Plan-and-Solve  steps predictable, want an auditable plan   batch reconciliation, KYC checklist
Reflexion       wrong answer is costly, needs completeness  compliance answers, fraud case summary
LATS            high-stakes, explore alternatives, can pay  complex underwriting, multi-option planning


☝ **LATS** (Language Agent Tree Search) generalises ReAct: it explores *multiple* trajectories as a tree and keeps the best-scoring branch — highest quality, highest cost, reserved for high-stakes decisions like underwriting. For most Barclays workloads the answer is ReAct (unknown path) or Plan-and-Solve (known path), with Reflexion bolted on where completeness is non-negotiable.

## 6. Recap — the reasoning layer is a design choice

| Pattern | Loop | Strength | Cost |
|---|---|---|---|
| **ReAct** | reason→act→observe, repeat | adapts mid-task | 1 call/step |
| **Plan-and-Solve** | plan all → execute | auditable, fewer calls | can't re-plan |
| **Reflexion** | act→critique→retry | self-corrects, complete | extra passes |
| **LATS** | tree search over ReAct | highest quality | most expensive |

Every pattern is the same tools under a different **control flow** — and in LangGraph each is a graph shape you already know (ReAct = loop with a tool node; Plan-and-Solve = plan node → executor; Reflexion = a critique edge back to the start). Yesterday's `BaseAgent(ABC)` is where these live: each pattern is one implementation of `run()`.